# 15.3 — Extended timeframe companion

This notebook validates the completed market snapshot and the canonical
static-window and trio table families before rendering. It does not write canonical
tables or mirrors.

Source-root override:
- `FINANCE_NOTEBOOK_SOURCE_ROOT`
- optional direct pin: `FINANCE_NOTEBOOK_MARKET_SNAPSHOT_DIR`

Presentation outputs only:
- `data/nb15_3_long_window_equity.png`
- `data/nb15_3_sharpe_vs_window.png`


In [ ]:
import hashlib
import io
import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _repository_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


REPO = _repository_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))

import matplotlib.pyplot as plt
import pandas as pd

from macro_framework.reporting import report_table
from scripts import build_basket_long as basket_producer
from scripts import build_static_bh as static_bh_producer
from scripts import build_tear_sheet as bts
from scripts import build_sjm_crowding as sjm
from scripts import extend_stream_2026 as ext


SNAPSHOT_ID_CANDIDATES = (
    "market_total_return_fx_2026-06-30_v1",
    "provisional_market_total_return_fx_2026-06-30_v1",
)
FACTOR_RUN_ID_CANDIDATES = (
    "factor_ext2026_2019-01-01_2026-06-30_v1",
)
SJM_RUN_ID_CANDIDATES = (
    "sjm_crowding_v3_total_return_bil",
)


def _resolve_override(value: str | None) -> Path | None:
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = (REPO / path).resolve()
    return path


def _search_roots() -> list[Path]:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_SOURCE_ROOT"))
    bases = [override] if override is not None else [
        REPO / "release_assets" / "data-v4",
        REPO / "data" / "provisional_remediation",
        REPO / "data",
        REPO,
    ]
    roots: list[Path] = []
    for base in bases:
        if base is None:
            continue
        roots.append(base)
        data_base = base / "data"
        if data_base != base:
            roots.append(data_base)
    deduped: list[Path] = []
    seen: set[str] = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            deduped.append(root)
    return deduped


SEARCH_ROOTS = _search_roots()


def find_existing_path(relative_candidates: list[str | Path]) -> Path | None:
    rels = [Path(rel) for rel in relative_candidates]
    for root in SEARCH_ROOTS:
        for rel in rels:
            candidate = root / rel
            if candidate.exists():
                return candidate
    return None


def _find_path(relative_candidates: list[str | Path]) -> Path:
    path = find_existing_path(relative_candidates)
    if path is not None:
        return path
    rels = [Path(rel) for rel in relative_candidates]
    return SEARCH_ROOTS[0] / rels[0]


def load_frame(path: Path, **csv_kwargs) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return pd.read_csv(path, **csv_kwargs)
    raise ValueError(f"unsupported table file type: {path}")


def resolve_snapshot_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_MARKET_SNAPSHOT_DIR"))
    if override is not None:
        return override
    rels = []
    for snapshot_id in SNAPSHOT_ID_CANDIDATES:
        rels.extend(
            [
                Path("market_snapshots") / snapshot_id,
                Path("provisional_remediation") / "market_snapshots" / snapshot_id,
            ]
        )
    return _find_path(rels)


def resolve_factor_run_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_FACTOR_RUN_DIR"))
    if override is not None:
        return override
    rels = []
    for run_id in FACTOR_RUN_ID_CANDIDATES:
        rels.extend(
            [
                Path("factor_runs") / run_id,
                Path("provisional_remediation") / "factor_runs" / run_id,
            ]
        )
    return _find_path(rels)


def resolve_sjm_run_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_SJM_RUN_DIR"))
    if override is not None:
        return override
    rels = []
    for run_id in SJM_RUN_ID_CANDIDATES:
        rels.extend(
            [
                Path("sjm_runs") / run_id,
                Path("provisional_remediation") / "sjm_runs" / run_id,
            ]
        )
    return _find_path(rels)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_manifest(run_dir: Path) -> dict[str, object]:
    return json.loads((run_dir / "manifest.json").read_text())


def read_parquet_inventoried(run_dir: Path, manifest: dict[str, object], key: str) -> pd.DataFrame:
    entry = manifest["files"][key]
    path = run_dir / entry["file"]
    actual = sha256_file(path)
    assert actual == entry["sha256"], f"{path} mutated after inventory"
    return pd.read_parquet(path)


def load_market_input(snapshot_dir: Path):
    manifest = read_manifest(snapshot_dir)
    manifest_sha256 = sha256_file(snapshot_dir / "manifest.json")
    basket_producer.validate_market_snapshot(snapshot_dir)
    market_input = bts.load_market_report_input(
        snapshot_dir,
        snapshot_id=manifest["snapshot_id"],
        manifest_sha256=manifest_sha256,
    )
    return market_input, manifest, manifest_sha256


def load_factor_input(run_dir: Path):
    manifest = read_manifest(run_dir)
    manifest_sha256 = sha256_file(run_dir / "manifest.json")
    try:
        factor_input = bts.load_factor_report_input(
            run_dir,
            run_id=manifest["run_id"],
            manifest_sha256=manifest_sha256,
        )
    except ValueError as exc:
        if "price_" not in str(exc):
            raise
        assert manifest.get("schema") == "factor_run.v1", manifest.get("schema")
        assert (run_dir / "COMPLETED").is_file(), f"{run_dir} is incomplete"
        for entry in manifest["files"].values():
            artifact_path = run_dir / entry["file"]
            assert artifact_path.is_file(), f"missing inventoried artifact: {artifact_path}"
            assert sha256_file(artifact_path) == entry["sha256"], f"{artifact_path} mutated after inventory"
        entry = manifest["files"]["metric_records"]
        metric_path = run_dir / entry["file"]
        metric_records = json.loads(metric_path.read_text())
        assert metric_records.get("schema") == "factor_run.metric_records.v1", metric_records.get("schema")
        factor_input = bts.VerifiedFactorRun(
            run_dir=run_dir,
            run_id=manifest["run_id"],
            manifest_sha256=manifest_sha256,
            manifest=manifest,
            metric_records=metric_records,
        )
    return factor_input, manifest, manifest_sha256


def load_sjm_input(run_dir: Path):
    manifest = read_manifest(run_dir)
    manifest_sha256 = sha256_file(run_dir / "manifest.json")
    sjm_input = bts.load_sjm_report_input(
        run_dir,
        run_id=manifest["run_id"],
        manifest_sha256=manifest_sha256,
    )
    return sjm_input, manifest, manifest_sha256


def active_value(value: pd.Series) -> pd.Series:
    moving = value[value.ne(value.iloc[0])]
    if moving.empty:
        return value
    first_move = moving.index.min()
    prior = value.index[value.index < first_move]
    start = prior.max() if len(prior) else first_move
    return value.loc[start:]


_DATE_COLUMNS = {
    "start",
    "end",
    "actual_end",
    "anchor",
    "first_return_date",
    "requested_start",
    "requested_end",
    "raw_market_model_start",
    "raw_market_model_end",
}


def normalize_table(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.columns:
        if column in _DATE_COLUMNS or column.endswith("_date"):
            out[column] = pd.to_datetime(out[column], errors="ignore")
    return out


def dataframe_sha256(frame: pd.DataFrame) -> str:
    payload = normalize_table(frame).to_json(orient="table", date_format="iso", index=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def csv_sha256(frame: pd.DataFrame, locale: str = "en-US") -> str:
    spec = dict(bts.REPORT_CSV_LOCALE_SPECS[locale])
    buffer = io.StringIO()
    writer = getattr(normalize_table(frame), "to_csv")
    writer(
        buffer,
        index=False,
        sep=spec["sep"],
        decimal=spec["decimal"],
        float_format=spec["float_format"],
    )
    return hashlib.sha256(buffer.getvalue().encode(spec["encoding"])).hexdigest()


## 1. Completed snapshot plus canonical static-window and trio tables


In [ ]:
configured_report_root = os.environ.get("FINANCE_NOTEBOOK_REPORT_ROOT")
REPORT_ROOT = _resolve_override(configured_report_root) if configured_report_root else None
if REPORT_ROOT is None:
    raise ValueError("set FINANCE_NOTEBOOK_REPORT_ROOT to a completed canonical_reports.v1 directory")
configured_output_dir = os.environ.get("FINANCE_NOTEBOOK_OUTPUT_DIR")
OUTPUT_DIR = _resolve_override(configured_output_dir) if configured_output_dir else REPO / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

report_manifest_path = REPORT_ROOT / "manifest.json"
report_completed_path = REPORT_ROOT / "COMPLETED"
assert report_manifest_path.is_file() and report_completed_path.is_file()
report_manifest = json.loads(report_manifest_path.read_text())
report_manifest_sha = sha256_file(report_manifest_path)
assert report_manifest.get("schema") == "canonical_reports.v1"
assert report_manifest.get("completed") is True
assert f"manifest_sha256={report_manifest_sha}" in report_completed_path.read_text().splitlines()

In [ ]:
MARKET_SNAPSHOT_DIR = resolve_snapshot_dir()
market_input, market_manifest, market_sha = load_market_input(MARKET_SNAPSHOT_DIR)

STATIC_SPECS = (
    bts.StaticWindowSpec("Full 16.7y (buy 2009)", pd.Timestamp("2009-09-25"), pd.Timestamp("2026-05-29")),
    bts.StaticWindowSpec("16.4y (buy 2009)", pd.Timestamp("2009-09-25"), pd.Timestamp("2026-01-30")),
    bts.StaticWindowSpec("10.0y (buy 2016)", pd.Timestamp("2016-02-01"), pd.Timestamp("2026-01-30")),
    bts.StaticWindowSpec("7.1y (buy 2019)", pd.Timestamp("2019-01-02"), pd.Timestamp("2026-01-30")),
)


def static_reader_view(frame: pd.DataFrame) -> pd.DataFrame:
    readers = frame[frame["schema"] == "portfolio_metrics.reader.v2"].copy()
    assert len(readers) == len(STATIC_SPECS), "canonical static table must contain one reader row per window"
    return pd.DataFrame(
        {
            "Label": readers["window_label"],
            "Start": pd.to_datetime(readers["start"]).dt.date,
            "End": pd.to_datetime(readers["end"]).dt.date,
            "Years (elapsed)": (
                (pd.to_datetime(readers["end"]) - pd.to_datetime(readers["start"]))
                .dt.days
                / 365.25
            ),
            "Number of days": readers["n_obs"].astype(int),
            "Total Return": readers["total_return"].astype(float),
            "CAGR": readers["cagr"].astype(float),
            "Ann. Volatility": readers["ann_vol"].astype(float),
            "Sharpe": readers["sharpe"].astype(float),
            "Sortino": readers["sortino"].astype(float),
            "Max Drawdown": readers["maxdd"].astype(float),
            "Calmar": readers["calmar"].astype(float),
            "SSR (Sharpe Stability Ratio)": readers["ssr_ssr"].astype(float),
            "Source": readers["source"],
        }
    )


def manifest_mirror_path(name: str) -> Path:
    entry = report_manifest["mirrors"].get(name)
    assert isinstance(entry, dict), f"canonical mirror inventory is missing {name}"
    path = (REPORT_ROOT / entry["file"]).resolve()
    assert path.is_relative_to(REPORT_ROOT.resolve())
    assert path.is_file() and sha256_file(path) == entry["sha256"]
    return path


static_us_path = manifest_mirror_path("tear_sheet_static_bh_windows.csv")
static_de_path = manifest_mirror_path("tear_sheet_static_bh_windows_de.csv")
trio_path = manifest_mirror_path("tear_sheet_trio_ext2026.csv")

static_us_loaded = pd.read_csv(static_us_path)
static_us = static_reader_view(static_us_loaded).set_index("Label")
static_de_loaded = pd.read_csv(static_de_path, sep=";", decimal=",")
static_de = static_reader_view(static_de_loaded).set_index("Label")
trio_loaded = pd.read_csv(trio_path)
trio_readers = trio_loaded[trio_loaded["schema"] == "portfolio_metrics.reader.v2"].copy()
assert len(trio_readers) == 3, f"trio mirror must carry exactly 3 reader rows, found {len(trio_readers)}"

static_rows = [bts.build_static_bh_rows(market_input, spec)[0] for spec in STATIC_SPECS]
expected_static = static_reader_view(report_table(static_rows)).set_index("Label")

for label in expected_static.index:
    for column in [
        "Start",
        "End",
        "Number of days",
        "Total Return",
        "CAGR",
        "Ann. Volatility",
        "Sharpe",
        "Sortino",
        "Max Drawdown",
        "Calmar",
        "SSR (Sharpe Stability Ratio)",
    ]:
        left = static_us.loc[label, column]
        right = expected_static.loc[label, column]
        if isinstance(left, float):
            assert abs(float(left) - float(right)) < 5e-9, (label, column, left, right)
        else:
            assert left == right, (label, column, left, right)
for label in static_us.index:
    for column in [
        "Start",
        "End",
        "Number of days",
        "Total Return",
        "CAGR",
        "Ann. Volatility",
        "Sharpe",
        "Sortino",
        "Max Drawdown",
        "Calmar",
        "SSR (Sharpe Stability Ratio)",
    ]:
        left = static_us.loc[label, column]
        right = static_de.loc[label, column]
        if isinstance(left, float):
            assert abs(float(left) - float(right)) < 5e-9, (label, column, left, right)
        else:
            assert left == right, (label, column, left, right)

trio_static = trio_readers[trio_readers["portfolio_id"].str.startswith("static_bh_")]
assert len(trio_static) == 1, "canonical trio must contain one static comparison row"
trio_static = trio_static.iloc[0]
assert trio_static["window_label"] == "Factor performance window (buy 2019-01-02)"
assert pd.Timestamp(trio_static["start"]) == pd.Timestamp("2019-01-03")
assert pd.Timestamp(trio_static["end"]) == pd.Timestamp("2026-06-30")
assert int(trio_static["n_obs"]) > 0
assert trio_static["cash_benchmark_id"] == f"BIL@{market_manifest['snapshot_id']}"
assert trio_static["currency_basis"] == "legacy_mixed_local_quotes"

trio_ids = trio_readers["portfolio_id"].tolist()

display(static_us[[
    "Start",
    "End",
    "Years (elapsed)",
    "Number of days",
    "Total Return",
    "CAGR",
    "Ann. Volatility",
    "Sharpe",
    "Max Drawdown",
    "Calmar",
    "SSR (Sharpe Stability Ratio)",
]])

print("market snapshot:", market_manifest["snapshot_id"], market_sha)
print("canonical report root:", REPORT_ROOT, report_manifest_sha)
print("static windows (US):", static_us_path, sha256_file(static_us_path))
print("static windows (DE):", static_de_path, sha256_file(static_de_path))
print("trio table path:", trio_path, sha256_file(trio_path))
print("trio portfolio IDs:", trio_ids)

## 2. The long line, and the region where the AI lines do not exist


In [ ]:
static_levels_all = pd.concat(
    [
        pd.read_parquet(MARKET_SNAPSHOT_DIR / "basket_adjusted_close_local.parquet")[["SWDA.L", "XLK", "IAU"]],
        pd.read_parquet(MARKET_SNAPSHOT_DIR / "cash_market_total_return.parquet")[["BIL"]],
    ],
    axis=1,
).dropna()
long_levels = static_levels_all.loc[pd.Timestamp("2009-09-25") : pd.Timestamp("2026-05-29")]
long_line = (0.25 * (long_levels / long_levels.iloc[0]).sum(axis=1)).rename("value")
cut = pd.Timestamp("2019-01-03")

fig, (ax_eq, ax_dd) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax_eq.plot(long_line.index, 100 * long_line / long_line.iloc[0], color="#2f6db3", lw=1.8)
ax_eq.set_yscale("log")
ax_eq.set_ylabel("equity (start = 100, log scale)")
ax_eq.set_title("Extended timeframe — static buy & hold only", fontsize=11)
ax_eq.grid(alpha=0.25)

drawdown = long_line / long_line.cummax() - 1.0
ax_dd.plot(drawdown.index, 100 * drawdown, color="#2f6db3", lw=1.4)
ax_dd.set_ylabel("drawdown (%)")
ax_dd.grid(alpha=0.25)

for axis in (ax_eq, ax_dd):
    axis.axvspan(long_line.index.min(), cut, color="#9a9a9a", alpha=0.16, zorder=0)
    axis.axvline(cut, color="#555555", lw=1.0, ls=":")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "nb15_3_long_window_equity.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Sharpe against window length


In [ ]:
window_view = static_us.copy()
spec_years = {spec.label: (spec.end - spec.start).days / 365.25 for spec in STATIC_SPECS}
window_view["Years (elapsed)"] = [spec_years[label] for label in window_view.index]

rung_labels = ["7.1y (buy 2019)", "10.0y (buy 2016)", "16.4y (buy 2009)"]
rungs = window_view.loc[rung_labels]
full = window_view.loc["Full 16.7y (buy 2009)"]

fig, ax = plt.subplots(figsize=(7.6, 4.0))
ax.plot(rungs["Years (elapsed)"], rungs["Sharpe"], "-o", color="#2f6db3", lw=1.8, ms=8)
for label, row in rungs.iterrows():
    ax.annotate(
        f"{row['Sharpe']:.2f}  {label}",
        (row["Years (elapsed)"], row["Sharpe"]),
        xytext=(10, 6),
        textcoords="offset points",
        fontsize=8.5,
    )
ax.scatter(full["Years (elapsed)"], full["Sharpe"], s=95, marker="D", color="#e8710a", edgecolor="white", linewidth=1.4)
ax.annotate(f"{full['Sharpe']:.2f}", (full["Years (elapsed)"], full["Sharpe"]), xytext=(10, -6), textcoords="offset points", fontsize=8.5)
ax.set_xlabel("window length (elapsed years)")
ax.set_ylabel("Sharpe")
ax.grid(alpha=0.25)
ax.set_title("The longer the window, the smaller the Sharpe", fontsize=11)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "nb15_3_sharpe_vs_window.png", dpi=300, bbox_inches="tight")
plt.show()

display(window_view[[
    "Start",
    "End",
    "Years (elapsed)",
    "Number of days",
    "Total Return",
    "CAGR",
    "Ann. Volatility",
    "Sharpe",
    "Max Drawdown",
    "Calmar",
    "SSR (Sharpe Stability Ratio)",
]])